# Fake News Detection Experiments

This notebook runs 10 different architectural setups combining Graph Neural Networks (GCN, GraphSAGE), Text Encoders, Co-Attention (CMCG), and Sentiment Analysis.

Results are automatically logged to a local file: `fake_news_detection_results.csv`.

In [3]:
# Instalamos las dependencias necesarias en el entorno del kernel actual
!pip install torch_geometric PyYAML pandas

In [4]:
import itertools
import pandas as pd
import time
import torch
import random
import numpy as np
import os

# Import the run_experiment function from our main module
from main import run_experiment

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def main():
    results_file = "fake_news_detection_results.csv"
    
    # Set up the CSV file and headers if it doesn't exist
    if not os.path.exists(results_file):
        df_init = pd.DataFrame(columns=[
            "Seed", "Dataset", "Feature", "Use GNN", "GNN Type", 
            "Use Text", "Use CMCG", "Use Sentiment", "Epochs", "Accuracy"
        ])
        df_init.to_csv(results_file, index=False)
        print(f"Created {results_file} for logging results.")
    else:
        print(f"Appending to existing {results_file}.")
    
    # Define the test matrix
    seeds = [42, 2026]
    datasets = ["gossipcop"] # We can add "politifact" later
    features = ["bert"] # We can add "spacy" later
    
    # Combinations of model architectures
    # Each tuple: (use_gnn, gnn_type, use_text, use_cmcg, use_sentiment)
    model_configs = [
        # 1. Baseline Text only
        (False, "GCN", True, False, False),
        
        # 2-3. Baseline GNN only (GCN & SAGE)
        (True, "GCN", False, False, False),
        (True, "SAGE", False, False, False),
        
        # 4-5. GNN + Text (Early fusion / Concatenation)
        (True, "GCN", True, False, False),
        (True, "SAGE", True, False, False),
        
        # 6-7. CMCG (Co-Attention between GNN and Text)
        (True, "GCN", True, True, False),
        (True, "SAGE", True, True, False),
        
        # 8. Text + Sentiment
        (False, "GCN", True, False, True),
        
        # 9. Early Fusion Multi-modal (SAGE + Text + Sentiment)
        (True, "SAGE", True, False, True),
        
        # 10. Full Model: GNN + Text + CMCG + Sentiment
        (True, "SAGE", True, True, True),
    ]

    for seed in seeds:
        for dataset in datasets:
            for feature in features:
                for use_gnn, gnn_type, use_text, use_cmcg, use_sentiment in model_configs:
                    set_seed(seed)
                    
                    config = {
                        "data": {
                            "dataset": dataset,
                            "feature": feature,
                            "batch_size": 128,
                            "data_dir": "dataset"
                        },
                        "model": {
                            "gnn_type": gnn_type,
                            "hidden_channels": 128,
                            "use_gnn": use_gnn,
                            "use_text": use_text,
                            "use_cmcg": use_cmcg,
                            "use_sentiment": use_sentiment,
                            "sentiment_dim": 1 # Dummy dimension
                        },
                        "training": {
                            "lr": 0.001,
                            "weight_decay": 0.01,
                            "epochs": 30, # Set to 30 for faster local testing
                            "device": "auto"
                        }
                    }
                    
                    print(f"\n--- Running Experiment ---")
                    print(f"Seed: {seed}, Dataset: {dataset}, Feature: {feature}")
                    print(f"GNN: {use_gnn} ({gnn_type}), Text: {use_text}, CMCG: {use_cmcg}, Sentiment: {use_sentiment}")
                    
                    try:
                        acc = run_experiment(config)
                        acc_val = float(acc)
                        print(f"Achieved Accuracy: {acc_val:.4f}")
                        
                        # Log to CSV
                        row_df = pd.DataFrame([{
                            "Seed": seed, "Dataset": dataset, "Feature": feature, 
                            "Use GNN": use_gnn, "GNN Type": gnn_type, "Use Text": use_text, 
                            "Use CMCG": use_cmcg, "Use Sentiment": use_sentiment, 
                            "Epochs": config['training']['epochs'], "Accuracy": acc_val
                        }])
                        row_df.to_csv(results_file, mode='a', header=False, index=False)
                        
                    except Exception as e:
                        print(f"Experiment failed: {e}")
                        row_df = pd.DataFrame([{
                            "Seed": seed, "Dataset": dataset, "Feature": feature, 
                            "Use GNN": use_gnn, "GNN Type": gnn_type, "Use Text": use_text, 
                            "Use CMCG": use_cmcg, "Use Sentiment": use_sentiment, 
                            "Epochs": config['training']['epochs'], "Accuracy": f"ERROR: {str(e)}"
                        }])
                        row_df.to_csv(results_file, mode='a', header=False, index=False)

if __name__ == "__main__":
    main()


Appending to existing fake_news_detection_results.csv.

--- Running Experiment ---
Seed: 42, Dataset: gossipcop, Feature: bert
GNN: False (GCN), Text: True, CMCG: False, Sentiment: False
Using device: mps
Epoch: 01, Loss: 0.6635, Train: 0.6978, Val: 0.7070, Test: 0.6916
Epoch: 02, Loss: 0.5957, Train: 0.7244, Val: 0.7198, Test: 0.6939
Epoch: 03, Loss: 0.5603, Train: 0.7363, Val: 0.7436, Test: 0.7209
Epoch: 04, Loss: 0.5324, Train: 0.7463, Val: 0.7509, Test: 0.7201
Epoch: 05, Loss: 0.5293, Train: 0.7482, Val: 0.7527, Test: 0.7224
Epoch: 06, Loss: 0.5072, Train: 0.7427, Val: 0.7308, Test: 0.7164
Epoch: 07, Loss: 0.5044, Train: 0.7564, Val: 0.7473, Test: 0.7282
Epoch: 08, Loss: 0.4938, Train: 0.7692, Val: 0.7582, Test: 0.7321
Epoch: 09, Loss: 0.4844, Train: 0.7720, Val: 0.7656, Test: 0.7287
Epoch: 10, Loss: 0.4772, Train: 0.7647, Val: 0.7344, Test: 0.7229
Epoch: 11, Loss: 0.4823, Train: 0.7491, Val: 0.7271, Test: 0.6984
Epoch: 12, Loss: 0.4731, Train: 0.7701, Val: 0.7308, Test: 0.7101
Epo

KeyboardInterrupt: 